In [1]:
# Cell 0: imports & config

import os
from pathlib import Path
import copy
import numpy as np
import torch

from pcdet.config import cfg, cfg_from_yaml_file
from pcdet.models import build_network, load_data_to_gpu
from pcdet.datasets import build_dataloader
from pcdet.utils import common_utils
import random


/storage/home/hcoda1/9/spanse30/.conda/envs/mmlab/lib/python3.9/site-packages/spconv/pytorch/functional.py:47: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  _TORCH_CUSTOM_FWD = amp.custom_fwd(cast_inputs=torch.float16)
/storage/home/hcoda1/9/spanse30/.conda/envs/mmlab/lib/python3.9/site-packages/spconv/pytorch/functional.py:97: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/storage/home/hcoda1/9/spanse30/.conda/envs/mmlab/lib/python3.9/site-packages/spconv/pytorch/functional.py:163: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, grad_output):
/storage/home/hcoda1/9/spanse30/.conda/envs/mmlab/lib/python3.9/site-packages/spconv/pytorch/functional.py:243: Futur

In [2]:
# Cell 1: load CenterPoint Waymo config

CFG_FILE = 'tools/cfgs/waymo_models/centerpoint.yaml' 
CKPT = 'output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth'  # <- put your ckpt here

cfg_from_yaml_file(CFG_FILE, cfg)
cfg.TAG = Path(CFG_FILE).stem
cfg.EXP_GROUP_PATH = 'centerpoint_waymo_demo'

logger = common_utils.create_logger()
logger.info(f'Loaded cfg from {CFG_FILE}')


2025-12-23 03:10:37,482   INFO  Loaded cfg from tools/cfgs/waymo_models/centerpoint.yaml


In [3]:
# Cell 2: build dataloader (Waymo, test split, batch_size=1)

test_set, test_loader, _ = build_dataloader(
    dataset_cfg=cfg.DATA_CONFIG,
    class_names=cfg.CLASS_NAMES,
    batch_size=1,
    dist=False,
    workers=4,
    logger=logger,
    training=False
)

len_test = len(test_set)
logger.info(f'Test set length: {len_test}')

2025-12-23 03:10:39,643   INFO  Loading Waymo dataset
2025-12-23 03:10:50,765   INFO  Total skipped info 0
2025-12-23 03:10:50,765   INFO  Total samples for Waymo dataset: 38597
2025-12-23 03:10:50,767   INFO  Test set length: 38597


In [4]:
model = build_network(
    model_cfg=cfg.MODEL,
    num_class=len(cfg.CLASS_NAMES),
    dataset=test_set
)
orig_forward = model.dense_head.forward

def forward_capture_raw(data_dict):
    # run original forward
    ret = orig_forward(data_dict)

    # capture RAW tensors BEFORE decode
    # these exist regardless of training/eval
    model.dense_head._raw_outputs = {
        k: v for k, v in model.dense_head.forward_ret_dict.items()
        if k != 'pred_dicts'
    }
    return ret

model.dense_head.forward = forward_capture_raw

logger.info(f'Loading checkpoint from: {CKPT}')
model.load_params_from_file(filename=CKPT, logger=logger, to_cpu=False)
model.cuda()
model.eval()
for p in model.parameters():
    p.requires_grad_(True)  # IMPORTANT: enable autograd through custom ops

2025-12-23 03:10:54,961   INFO  Loading checkpoint from: output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth
2025-12-23 03:10:54,963   INFO  ==> Loading parameters from checkpoint output/cfgs/custom_models/centerpoint_singleframe_waymo/default/ckpt/checkpoint_epoch_30.pth to GPU
/storage/project/r-gchou3-0/spanse30/OpenPCDet/pcdet/models/detectors/detector3d_template.py:367: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they

In [5]:
# Cell 4: pull a single batch (one Waymo frame)

data_iter = iter(test_loader)
batch_dict = next(data_iter)

load_data_to_gpu(batch_dict)

# Sanity check shapes
print('points:', batch_dict['points'].shape)         # [N, 5] -> [batch_idx, x, y, z, i]
print('voxels:', batch_dict['voxels'].shape)         # [V, T, C]
print('voxel_coords:', batch_dict['voxel_coords'].shape)  # [V, 4] -> [b, z, y, x]
print('voxel_num_points:', batch_dict['voxel_num_points'].shape)  # [V]


points: torch.Size([126501, 6])
voxels: torch.Size([75604, 5, 5])
voxel_coords: torch.Size([75604, 4])
voxel_num_points: torch.Size([75604])


In [6]:
from pcdet.utils import box_utils

def init_spoof_spherical(xa,ya,za, la, omega_min_offset, n, sector_deg=8.0, device = 'cuda', random_ = False):

    base_alpha = np.arctan2(ya, xa) # azimuth of bbox cente_r
    # print(za)
    base_alpha = 0
    # print(f"base alpha rad: {base_alpha}, base alpha deg: {np.rad2deg(base_alpha)}") 
    
    #!uniformly generate points in sector (range of azimuths)
    # ensure that there is at most one return per ray
    alphas = np.linspace(base_alpha - np.deg2rad(sector_deg/2),
                         base_alpha + np.deg2rad(sector_deg/2), n)
    
    # print(np.rad2deg(alphas))
    
    R0 = np.sqrt(xa*xa+ya*ya+za*za)
    # print(R0)
    # print(np.floor(np.rad2deg(np.arcsin(za/R0))) - omega_min_offset)
    omega_min = np.floor(np.rad2deg(np.arcsin(za/R0)))-omega_min_offset
    # print(np.rad2deg(omega_min))
    omega_set = np.deg2rad(np.linspace(omega_min, 5, int(6 - omega_min))) 
        
    if random_:
        #for each point, randomly select an elevation from permissible set
        omegas = np.array([random.choice(omega_set) for i in range(0, n)])
        # print(f"omegas rad: {omegas}, omegas deg: {np.rad2deg(omegas)}")
        stddev = la/4
        R = torch.tensor(np.clip(np.random.normal(R0, stddev, n), R0-la/2, R0+la/2),
                          dtype=torch.float32, device=device)
        # print(omegas)
    else:
        omega_set = np.deg2rad(np.linspace(-24.9, 2, 64)) #vertical FOV for Velodyne HDL-64E . np.linspace(-24.9, 2, 64)
        omegas = omega_set[np.linspace(0, 63, n, dtype=int)]  # evenly spaced channels
        R = torch.tensor(np.linspace(R0-0.9, R0+0.9, n), dtype=torch.float32)

        
    # print(f"R tensor: {R}")
    return torch.tensor(alphas, dtype=torch.float32, device=device), \
           torch.tensor(omegas, dtype=torch.float32, device=device), R

from pcdet.ops.iou3d_nms import iou3d_nms_utils

def iou_bev(boxes, ref7):
    """
    boxes: (K,7) predicted boxes
    ref7 : (7,) ground truth box
    returns (K,) bev iou
    """
    device = boxes.device

    ref = ref7.view(1, 7).to(device)

    # The CUDA code supports torch tensors directly
    ious = iou3d_nms_utils.boxes_iou_bev(boxes, ref)  # (K,1)

    return ious[:, 0]

In [7]:
"""
Convert spherical coordinates to xyz
omega = vertical angle #!define as angle between vector to point and  xy plane (not angle with z axis as conventional)
alpha = azimuth (ccw angle with x axis)
"""
def spherical_to_xyz(alpha, omega, R, xa, ya, za, la, ha, wa, yawa):
    device = R.device

    xa_t = torch.tensor(xa, device=device)
    ya_t = torch.tensor(ya, device=device)

    base_alpha = torch.atan2(ya_t, xa_t)
    R0_2D = torch.sqrt(xa_t*xa_t + ya_t*ya_t)

    y_adjustment = R0_2D - R0_2D * torch.cos(base_alpha)
    x_adjustment = R0_2D * torch.sin(base_alpha)

    x = R * torch.cos(omega) * torch.cos(alpha) - y_adjustment
    y = R * torch.cos(omega) * torch.sin(alpha) + x_adjustment
    z = R * torch.sin(omega)

    return torch.stack([x, y, z], dim=-1)

# def spherical_to_xyz(alpha, omega, R, xa,ya,za, la, ha, wa, yawa):
#     base_alpha = np.arctan2(ya, xa)# azimuth of bbox cente_r
#     # print(np.rad2deg(base_alpha))
#     R0_2D = np.sqrt(xa*xa+ya*ya)
#     # print(R0_2D * np.cos(base_alpha))
#     # print(R0_2D * np.sin(base_alpha))
    
#     y_adjustment = R0_2D*torch.ones(alpha.shape[0]) - R0_2D *np.cos(base_alpha) * torch.ones(alpha.shape[0])
#     y_adjustment = y_adjustment.to(alpha.device)
#     x_adjustment = R0_2D *np.sin(base_alpha) * torch.ones(alpha.shape[0]).to(alpha.device)
#     # print(y_adjustment[0])
#     # print(x_adjustment[0])
#     base_alpha = np.arctan2(ya, xa)# azimuth of bbox cente_r
#     x = R * torch.cos(omega) * torch.cos(alpha) - y_adjustment
#     y = R * torch.cos(omega) * torch.sin(alpha) + x_adjustment
#     z = R * torch.sin(omega)
#     return torch.stack([x, y, z], dim=-1)

"""give spoofed points a fixed intensity
TODO tune intensity value
"""
def make_intensities_elongation_like(points_xyz, value_i=0.5, value_e = 0.05):
    #count # of spoofed points
    N = points_xyz.shape[0]
    
    intensities = torch.full((N,1), float(value_i), device=points_xyz.device, dtype=points_xyz.dtype)
    elongation = torch.full((N,1), float(value_e), device=points_xyz.device, dtype=points_xyz.dtype)
    #return Nx1 tensor  of intensities 
    return torch.cat([intensities, elongation], dim = 1)


import copy
def inject_spoof(batch_dict, spoof_xyzi):
    """
    batch_dict : OpenPCDet batch (after load_data_to_gpu)
    spoof_xyzi : (N,4) tensor [x, y, z, intensity] — differentiable
    
    Returns:
        new_batch_dict : with spoofed points appended
        spoof_points_full : (N,5) tensor [batch_idx, x, y, z, intensity]
                            that carries the gradient
    """
    new_batch = {}
    for k, v in batch_dict.items():
        if torch.is_tensor(v):
            new_batch[k] = v.clone()
        else:
            new_batch[k] = copy.deepcopy(v)
    device = new_batch['points'].device
    spoof_xyz = spoof_xyzi.to(device)
    # print(new_batch['points'].shape)
    # print(type(spoof_xyzi))
    # spoof_xyzi.requires_grad(True)

    batch_idx = 0
    # bcol = torch.full((spoof_xyzi.shape[0], 1), float(batch_idx), device = device)
    # spoof_full = torch.cat([bcol, spoof_xyzi], dim = 1)
    orig_points = new_batch['points']
    orig_points = orig_points.detach()
    new_points = torch.cat([orig_points, spoof_xyzi], dim = 0)
    new_batch['points'] = new_points
    
    spoof_id = torch.arange(orig_points.shape[0], spoof_xyz.shape[0] + orig_points.shape[0], device=device)
    return new_batch, spoof_id
     
    


def hiding_loss(logits, ious, eps_i=0.1, eps_s=0.1):
    """
    logits: (K,) raw class logits for proposals associated with the victim
    ious:   (K,) IoU (BEV or 3D) between each proposal and the victim GT (NO grad)

    eps_i: IoU threshold for relevance
    eps_s: classification confidence threshold (on sigmoid(logits))

    Returns:
        loss (scalar tensor)
        num_relevant (int)
    """
    scores = torch.sigmoid(logits)

    #proposals are relevant if it overlaps enough and is confident enough 

    rel = (ious >= eps_i) & (scores >= eps_s)
    # print(torch.sum(rel).item())
    #weight = non-negative IoU times relevance mask (only want relevant ious)
    #.detach--> ensure no gradients through IoU
    #high IoU proposals count more
    iou_rel = ious.clamp(min=0).detach()*rel.float()
    
    #calculate log(1-s) with softplus instead of scores (probabilities)
    # if score becomes too close to 1, log(0) will blow up
    #-log(1-s) = -log(1-sigma(z)) = log(1+e^z)
    # gamma = 2
    # neg_log_s = ((scores)**gamma) * torch.nn.functional.softplus(logits)
    neg_log_s = torch.nn.functional.softplus(logits)
   
    #rel.sum(): count of relevant proposals
    return (iou_rel *neg_log_s).sum(), rel.sum().item()

def map_points_to_voxels(model, batch_dict):
    """
    in:
        model      - OpenPCDet model (has model.dataset with voxel metadata)
        batch_dict - OpenPCDet batch after load_data_to_gpu

    out:
        point2voxel      - LongTensor [N_points], index into voxel_coords or -1
        voxel2points_trunc - list length V, each entry LongTensor of point indices (clipped to max_pts)
        slot_to_point    - LongTensor [V, max_pts], point index or -1
    """
    points = batch_dict['points'] # [N, 5] -> [b, x, y, z, i]
    voxels = batch_dict['voxels'] # [V, T, C]
    coords = batch_dict['voxel_coords'] # [V, 4] -> [b, z, y, x]
    device = points.device
    
    V, T, C = voxels.shape
    max_pts = T #per-voxel capacity from data processor
    
    #metadata
    ds = model.dataset
    voxel_size = torch.tensor(ds.voxel_size, dtype=torch.float32, device=device) # (vx, vy, vz)
    pc_range = torch.tensor(ds.point_cloud_range, dtype=torch.float32, device=device)
    gx, gy, gz = torch.floor((pc_range[3:6] - pc_range[0:3]) / voxel_size).long() # (gx, gy, gz)
    xyz = points[:, 1:4] # (x, y, z)
    idx_xyz = torch.floor((xyz - pc_range[:3]) / voxel_size).long() # [N, 3]
    in_range = (
        (idx_xyz >= 0).all(dim=1) &
        (idx_xyz[:, 0] < gx) &
        (idx_xyz[:, 1] < gy) &
        (idx_xyz[:, 2] < gz)
    )
    
    #build [b, z, y, x] coords for each point
    b_idx = points[:, 0].long() # batch_idx (all 0 if batch_Size = 1)
    idx_zyx = idx_xyz[:, [2, 1, 0]] # (z, y x)
    point_coords = torch.stack(
            [b_idx, idx_zyx[:, 0], idx_zyx[:, 1], idx_zyx[:, 2]],
            dim=1
        )  # [N,4] = [b,z,y,x]
    
    #input 4 column tensor of voxel coordinates
    # use a hash map instead of comparing every point to every voxel coordinate (O (NxV) comparisons)
    def hash_coords(c):
        #compute unique integer id--> pack 4 integers into 1 integer
        #batch offset +z - layer offset + y offset + x index
        """
            batch offset = ensure different batches get different large blocks of indices
            z layer offset = separates different height slices
            y offset = separates rows
            x index = column within row
        """
        return (c[:, 0] * (gz * gy * gx)
              + c[:, 1] * (gy * gx)
              + c[:, 2] * gx
              + c[:, 3])

    vox_hash   = hash_coords(coords)       # (V,)
    point_hash = hash_coords(point_coords) # (N,)
    sort_vals, sort_idx = torch.sort(vox_hash) #voxel hashes in ascending order, indices that tell how voxels were rearranged during sorting
    pos = torch.searchsorted(sort_vals, point_hash) #find where each point's hash fits in the sorted voxel  hashes
    matched = pos < sort_vals.numel() # boolean mask to ensure we dont go out of bounds when indexing later 
    
    #filter out points whose hashes dont exist in any voxel. All points should be voxelized though
    matched &= (sort_vals[pos.clamp_max(sort_vals.numel() - 1)] == point_hash)
    point2voxel = torch.full((points.shape[0],), -1, dtype=torch.long, device=device) # initialize array of length = num points, set to -1. -1 = point didnt match any voxel
    #point2voxel[1] = voxel row index. For each point, which voxel it belongs to 
    point2voxel[matched & in_range] = sort_idx[pos[matched & in_range]] # point i belongs to voxel index sort_idx[pos[i]] in voxel tensor

    valid = point2voxel>=0             #true for points that belong to a valid voxel
    voxel_ids = point2voxel[valid]     #pick voxel indices for valid points
    point_ids = torch.nonzero(valid, as_tuple = False).squeeze(1)       # find indices of valid points . returns list of positions where valid ==True
    counts = torch.zeros((V,), dtype=torch.long, device=device)         #one slot per voxel

    #vectorized method for counting how many points to into each voxel 
    #should match num_points_per_voxel
    if voxel_ids.numel()>0:   
        #for each voxel index in voxel_ids, add corresponding valie (1) to counts at that location
        # scatter_add_ : add all values from src (torch.ones_like(voxel_ids)) at indices specified in index (voxel_ids), along a dim (0)
        counts.scatter_add_(0, voxel_ids, torch.ones_like(voxel_ids, dtype = torch.long))

    voxel2points = [None]*V #each entry to store a tensor of point indices inside that voxel
    buckets = [[] for _ in range(V)]

    #build mapping on opposite direction: iterate thru each (point_id, voxel_id) pair. Append the point's index for a voxel id
    for pid, vid in zip(point_ids.tolist(), voxel_ids.tolist()):
        buckets[vid].append(pid)
    #turn python lists to PyTorch tenosrs
    voxel2points = [
        torch.tensor(b, dtype=torch.long, device=device)
        if b else torch.empty(0, dtype = torch.long, device = device) 
        for b in buckets
    ]
    voxel2points_trunc = [v[:max_pts] for v in voxel2points] # only keep first 32 points per voxel. trims any lists that exceed 32

    
    #each row refers to a voxel
    #each entry of each row is the index of the point in that voxel (-1 indicates all 32 points arent used up)
    slot_to_point = torch.full((V, max_pts), -1, dtype=torch.long, device=device)
    for v, ids in enumerate(voxel2points_trunc):
        n = min(int(ids.numel()), max_pts)
        if n > 0:
            #row at index v, up to but not including column n = slice of array up tobut not including index n 
            slot_to_point[v, :n] = ids[:n]
    # print(slot_to_point[0])
    return point2voxel, slot_to_point



def map_voxel_to_coords(model, batch_dict):
    """
    Build a dict mapping BEV grid cell (y_idx, x_idx) -> voxel_id.
    Ignores z index 

    Useful for quickly going from a BEV (y,x) cell to an index in voxel_coords.
    """
    coords = batch_dict['voxel_coords']  # [V,4] = [b,z,y,x]

    # Extract y,x,z for each voxel
    bz = coords[:, 1].tolist()
    by = coords[:, 2].tolist()
    bx = coords[:, 3].tolist()

    key2vid = {}

    for vid in range(coords.shape[0]):
        key2vid[(by[vid], bx[vid])] = vid

    return key2vid

In [8]:
def detach_to_cpu(batch_dict):
    """Simple PCDet version."""
    out = {}
    for k, v in batch_dict.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.detach().cpu()
        else:
            out[k] = v
    return out
def print_gpu_memory(label=""):
    allocated = torch.cuda.memory_allocated() / (1024**2)   # MB
    reserved = torch.cuda.memory_reserved() / (1024**2)     # MB
    max_alloc = torch.cuda.max_memory_allocated() / (1024**2)
    print(f"[{label}] Allocated: {allocated:.2f} MB | Reserved: {reserved:.2f} MB | Max Alloc: {max_alloc:.2f} MB")

In [9]:
def extract_prenms_from_centerpoint(model, batch_dict):
    """
    Returns:
        boxes  : (N, 7)
        scores : (N,)
        logits : (N,)
    """
    pred_dicts_ = model.dense_head.forward_ret_dict['pred_dicts']
    # This is REQUIRED: generate_predicted_boxes reads forward_ret_dict
    pred_dicts = model.dense_head.generate_predicted_boxes(
        batch['batch_size'],
        pred_dicts_     
    )

    # Single batch element
    pred = pred_dicts[0]

    boxes  = pred['pred_boxes']      # (N, 7)
    scores = pred['pred_scores']     # (N,)

    # Convert scores → logits safely
    eps = 1e-6
    scores_clamped = scores.clamp(eps, 1 - eps)
    logits = torch.log(scores_clamped / (1 - scores_clamped))

    return boxes, scores, logits

def forward_with_ste(model, batch_dict, spoof_tag=0.05):
    device = batch_dict['voxels'].device
    model.eval()

    # --------------------------------------------------
    # 1. STE injection at voxel level
    # --------------------------------------------------
    voxels = batch_dict['voxels']        # (V, T, C)

    tag_mask = voxels[..., 4] > (spoof_tag - 1e-6)

    spoof_proxy = torch.zeros_like(voxels, requires_grad=True)
    voxels_ste = voxels + tag_mask.unsqueeze(-1) * spoof_proxy

    batch_dict['voxels'] = voxels_ste

    # --------------------------------------------------
    # 2. Normal forward pass (NO post_processing)
    # --------------------------------------------------
    batch_dict = model.vfe(batch_dict)
    batch_dict = model.backbone_3d(batch_dict)
    batch_dict = model.map_to_bev_module(batch_dict)
    batch_dict = model.backbone_2d(batch_dict)
    batch_dict = model.dense_head(batch_dict)

    # --------------------------------------------------
    # 3. Decode PRE-NMS boxes (correct way)
    # --------------------------------------------------
    boxes, scores, logits = extract_prenms_from_centerpoint(model, batch_dict)

    return boxes, scores, logits, spoof_proxy


In [10]:
def ste_voxelize_mean(points_bxyzie, voxel_size, pc_range):
    """
    points_bxyzie: (N, 1+F) = [b, x, y, z, ...]
    returns:
      coords: (V,4) int [b,z,y,x]
      voxel_features: (V,F) float (STE: hard forward, soft backward)
    """
    device = points_bxyzie.device
    b = points_bxyzie[:, 0:1]
    xyz = points_bxyzie[:, 1:4]
    feats = points_bxyzie[:, 1:]  # include xyz in features so gradients can move points

    pc0 = torch.as_tensor(pc_range[:3], device=device, dtype=xyz.dtype)
    vs  = torch.as_tensor(voxel_size, device=device, dtype=xyz.dtype)

    u = (xyz - pc0) / vs                      # continuous voxel coord (x,y,z)
    u0 = torch.floor(u).detach()              # base index (detached)
    frac = (u - u0).clamp(0, 1)               # differentiable in-cell fraction

    # 8 corners offsets
    offs = torch.tensor(
        [[0,0,0],[1,0,0],[0,1,0],[1,1,0],[0,0,1],[1,0,1],[0,1,1],[1,1,1]],
        device=device, dtype=u0.dtype
    )  # (8,3)

    # neighbor integer coords (detached)
    nbr = (u0[:, None, :] + offs[None, :, :])     # (N,8,3) in xyz-index space
    # weights (trilinear)
    fx, fy, fz = frac[:, 0:1], frac[:, 1:2], frac[:, 2:3]
    w000 = (1-fx)*(1-fy)*(1-fz)
    w100 = (fx)*(1-fy)*(1-fz)
    w010 = (1-fx)*(fy)*(1-fz)
    w110 = (fx)*(fy)*(1-fz)
    w001 = (1-fx)*(1-fy)*(fz)
    w101 = (fx)*(1-fy)*(fz)
    w011 = (1-fx)*(fy)*(fz)
    w111 = (fx)*(fy)*(fz)
    w = torch.cat([w000,w100,w010,w110,w001,w101,w011,w111], dim=1)  # (N,8)

    # build coords for scatter: [b,z,y,x]
    # nbr is (x,y,z) index; convert to (z,y,x)
    nbr_zyx = nbr[..., [2,1,0]].long()
    b_long = b.long().expand(-1, 8)  # (N,8)
    coords_all = torch.stack([b_long, nbr_zyx[...,0], nbr_zyx[...,1], nbr_zyx[...,2]], dim=-1)  # (N,8,4)
    coords_all = coords_all.view(-1, 4)  # (N*8,4)

    # unique voxel rows
    uniq, inv = torch.unique(coords_all, dim=0, return_inverse=True)  # uniq: (V,4), inv: (N*8,)

    F = feats.shape[1]
    feats_rep = feats[:, None, :].expand(-1, 8, -1).reshape(-1, F)     # (N*8,F)
    w_rep = w.reshape(-1, 1)                                           # (N*8,1)

    # soft mean
    num = torch.zeros((uniq.shape[0], F), device=device, dtype=feats.dtype)
    den = torch.zeros((uniq.shape[0], 1), device=device, dtype=feats.dtype)
    num.index_add_(0, inv, feats_rep * w_rep)
    den.index_add_(0, inv, w_rep)
    voxel_soft = num / den.clamp(min=1e-6)

    # hard mean (forward exact): assign to base voxel only (offset 000)
    base = torch.stack([b.long().squeeze(1), u0[:,2].long(), u0[:,1].long(), u0[:,0].long()], dim=1)  # (N,4)
    uniq_h, inv_h = torch.unique(base, dim=0, return_inverse=True)
    num_h = torch.zeros((uniq_h.shape[0], F), device=device, dtype=feats.dtype)
    den_h = torch.zeros((uniq_h.shape[0], 1), device=device, dtype=feats.dtype)
    num_h.index_add_(0, inv_h, feats)
    den_h.index_add_(0, inv_h, torch.ones_like(inv_h, dtype=feats.dtype).unsqueeze(1))
    voxel_hard = num_h / den_h.clamp(min=1e-6)

    # Put hard into the same voxel table as soft (missing rows -> 0)
    # map uniq_h rows into uniq
    # (hash via unique match)
    # easiest: build dict on CPU once if V small; otherwise implement hashing like your cell-6 helper.
    key = {tuple(r.tolist()): i for i, r in enumerate(uniq.cpu())}
    vh = torch.zeros_like(voxel_soft)
    for i, r in enumerate(uniq_h.cpu()):
        j = key.get(tuple(r.tolist()), None)
        if j is not None:
            vh[j] = voxel_hard[i].to(device)

    voxel_feat = voxel_soft + (vh - voxel_soft).detach()
    return uniq, voxel_feat


In [18]:
import torch

def _get_voxel_cfg_from_dataset(model):
    ds = model.dataset
    voxel_size = torch.tensor(ds.voxel_size, device='cuda', dtype=torch.float32)          # (vx,vy,vz)
    pc_range   = torch.tensor(ds.point_cloud_range, device='cuda', dtype=torch.float32)  # (xmin,ymin,zmin,xmax,ymax,zmax)

    grid_size = torch.floor((pc_range[3:6] - pc_range[0:3]) / voxel_size).long()
    gx, gy, gz = grid_size.tolist()  # x,y,z

    # Pull max_points / max_voxels from cfg (works for Waymo CenterPoint configs)
    max_points = None
    max_voxels = None
    for dp in model.dataset.dataset_cfg.DATA_PROCESSOR:
        if dp.NAME == 'transform_points_to_voxels':
            max_points = int(dp.MAX_POINTS_PER_VOXEL)
            # PCDet uses different train/test; for inference use MAX_NUMBER_OF_VOXELS['test'] if dict
            mv = dp.MAX_NUMBER_OF_VOXELS
            max_voxels = int(mv['test']) if isinstance(mv, dict) else int(mv)
            break
    assert max_points is not None and max_voxels is not None, "Couldn't find voxel config in DATA_PROCESSOR"

    return voxel_size, pc_range, (gx,gy,gz), max_points, max_voxels


# @torch.no_grad()
def hard_voxelize_torch(points_bfe, voxel_size, pc_range, grid_xyz, max_points, max_voxels):
    """
    points_bfe: (N, 1+C) where col0=batch_idx, remaining are point features (x,y,z,i[,e])
    Returns:
      voxels_true: (V, T, C)
      voxel_coords: (V,4) [b,z,y,x]
      voxel_num_points: (V,)
      slot_to_point: (V,T) indices into ORIGINAL points_bfe rows, or -1
    """
    device = points_bfe.device
    dtype  = points_bfe.dtype
    gx,gy,gz = grid_xyz

    b   = points_bfe[:, 0].long()
    xyz = points_bfe[:, 1:4].float()
    feats = points_bfe[:, 1:].float()     # exclude batch_idx for voxel contents

    idx_xyz = torch.floor((xyz - pc_range[:3]) / voxel_size).long()  # (N,3) in x,y,z order

    in_range = (
        (idx_xyz[:, 0] >= 0) & (idx_xyz[:, 0] < gx) &
        (idx_xyz[:, 1] >= 0) & (idx_xyz[:, 1] < gy) &
        (idx_xyz[:, 2] >= 0) & (idx_xyz[:, 2] < gz)
    )

    # filter
    keep = torch.nonzero(in_range, as_tuple=False).squeeze(1)
    if keep.numel() == 0:
        # no points in range
        voxels_true = torch.zeros((0, max_points, feats.shape[1]), device=device, dtype=dtype)
        voxel_coords = torch.zeros((0,4), device=device, dtype=torch.int32)
        voxel_num_points = torch.zeros((0,), device=device, dtype=torch.int32)
        slot_to_point = torch.zeros((0, max_points), device=device, dtype=torch.long)
        return voxels_true, voxel_coords, voxel_num_points, slot_to_point

    b   = b[keep]
    idx = idx_xyz[keep]
    feats = feats[keep]
    orig_ids = keep  # map back to original points rows

    # coords = [b,z,y,x]
    coords = torch.stack([b, idx[:,2], idx[:,1], idx[:,0]], dim=1).to(torch.int64)

    # hash coords (int64) + tie-breaker to preserve input order inside a voxel
    hash_val = (coords[:,0] * (gz*gy*gx) +
                coords[:,1] * (gy*gx) +
                coords[:,2] * gx +
                coords[:,3])

    tie = torch.arange(hash_val.numel(), device=device, dtype=torch.int64)
    order = torch.argsort(hash_val * (tie.max()+1) + tie)  # stable grouping

    coords_s = coords[order]
    feats_s  = feats[order]
    orig_s   = orig_ids[order]
    hash_s   = hash_val[order]

    uniq, counts = torch.unique_consecutive(hash_s, return_counts=True)
    V = min(int(uniq.numel()), int(max_voxels))
    counts = counts[:V]

    starts = torch.cat([torch.zeros(1, device=device, dtype=torch.long),
                        counts.cumsum(0)[:-1]])

    C = feats_s.shape[1]
    voxels_true = torch.zeros((V, max_points, C), device=device, dtype=feats_s.dtype)
    voxel_coords = coords_s[starts].to(torch.int32)              # [V,4]
    voxel_num_points = torch.clamp(counts, max=max_points).to(torch.int32)

    slot_to_point = torch.full((V, max_points), -1, device=device, dtype=torch.long)

    for v in range(V):
        s = int(starts[v].item())
        e = int((starts[v] + counts[v]).item())
        k = min(e - s, max_points)
        if k <= 0:
            continue
        voxels_true[v, :k] = feats_s[s:s+k]
        slot_to_point[v, :k] = orig_s[s:s+k]

    return voxels_true, voxel_coords, voxel_num_points, slot_to_point


In [42]:
def build_voxels_ste(model, batch_points_diff):
    device = batch_points_diff.device
    voxel_size, pc_range, grid_xyz, max_points, max_voxels = _get_voxel_cfg_from_dataset(model)

    # HARD voxelization path (no grad needed)
    vox_true, voxel_coords, voxel_num_points, slot_to_point = hard_voxelize_torch(
        batch_points_diff.detach(), voxel_size, pc_range, grid_xyz, max_points, max_voxels
    )

    V, T, C = vox_true.shape
    valid = slot_to_point >= 0

    # Build packed OUT-OF-PLACE so it is differentiable w.r.t. batch_points_diff
    if valid.any():
        v_idx, t_idx = valid.nonzero(as_tuple=False).T
        p_idx = slot_to_point[valid]                      # indices into batch_points_diff
        src  = batch_points_diff[p_idx, 1:1+C].float()    # (K, C)

        flat_idx = v_idx * T + t_idx                      # (K,)
        packed_flat = torch.zeros((V * T, C), device=device, dtype=src.dtype)

        # out-of-place index_put is tracked by autograd
        packed_flat = packed_flat.index_put((flat_idx,), src, accumulate=False)
        packed = packed_flat.view(V, T, C)
    else:
        packed = torch.zeros((V, T, C), device=device, dtype=vox_true.dtype)

    vox_proxy = vox_true.detach() + (packed - packed.detach())
    return vox_proxy, voxel_coords, voxel_num_points, slot_to_point, packed

def _rg(x, name):
    if torch.is_tensor(x):
        print(f"{name}: requires_grad={x.requires_grad}, grad_fn={x.grad_fn}")
    else:
        print(f"{name}: <not a tensor>")



# def forward_with_ste_centerpoint_raw(model, base_batch_dict, points_diff):
#     # 1) STE voxelization
#     vox_proxy, voxel_coords, voxel_num_points, slot_to_point, packed = build_voxels_ste(model, points_diff)
#     packed.retain_grad()       # 🔴 REQUIRED
#     vox_proxy.retain_grad()    # (optional but useful)
    
#     spoof_start = points_diff.shape[0] - n  # last n points are spoof
#     mask = (slot_to_point >= spoof_start)

#     # how many spoof points landed in a voxel?
#     print("Spoof points voxelized:", mask.sum().item())

#     # how many spoof points are actually USED by MeanVFE?
#     used = mask & (torch.arange(slot_to_point.shape[1], device=slot_to_point.device)[None, :]
#                    < voxel_num_points[:, None])
#     print("Spoof points used in voxel_features:", used.sum().item())
    
#     # vox_proxy_scaled = vox_proxy.clone()
#     # vox_proxy_scaled[:, :, :3] *= 100.0
#     batch_dict = {
#         'batch_size': int(base_batch_dict.get('batch_size', 1)),
#         'points': points_diff,
#         'voxels': vox_proxy.contiguous().float(),      # THIS is the differentiable input
#         'voxel_coords': voxel_coords.contiguous().int(),
#         'voxel_num_points': voxel_num_points.contiguous().int(),
#     }

#     # 2) Backbone
#     batch_dict = model.vfe(batch_dict)
    
#     print("keys after vfe:", [k for k in batch_dict.keys() if "voxel" in k or "spatial" in k])
#     vf = batch_dict.get("voxel_features", None)
#     print("voxel_features:", None if vf is None else (vf.requires_grad, vf.grad_fn, vf.shape))

#     batch_dict = model.backbone_3d(batch_dict)
#     print("keys after backbone3d:", [k for k in batch_dict.keys()])
#     if 'encoded_spconv_tensor' in batch_dict:
#         print("Grad inside Backbone3D (SparseTensor):", batch_dict['encoded_spconv_tensor'].features.requires_grad)
#     batch_dict = model.map_to_bev_module(batch_dict)
#     batch_dict = model.backbone_2d(batch_dict)

#     # 3) Direct head calls (NO dense_head.forward, NO pred_dicts)
#     spatial_features_2d = batch_dict['spatial_features_2d']  # should require_grad=True

#     x = model.dense_head.shared_conv(spatial_features_2d)
#     head = model.dense_head.heads_list[0]
    
#     print("vox_proxy requires_grad:", vox_proxy.requires_grad, vox_proxy.grad_fn)
#     print("packed requires_grad:", packed.requires_grad, packed.grad_fn)
#     print("spatial_features_2d requires_grad:", spatial_features_2d.requires_grad, spatial_features_2d.grad_fn)
#     print("x requires_grad:", x.requires_grad, x.grad_fn)

    
#     hm_raw     = head.hm(x)       # (B, num_classes, H, W) logits
#     center_raw = head.center(x)
#     dim_raw    = head.dim(x)
#     rot_raw    = head.rot(x)

#     return hm_raw, center_raw, dim_raw, rot_raw, vox_proxy, packed




In [43]:
def dummy_hm_loss(model):
    frd = model.dense_head.forward_ret_dict
    pred0 = frd['pred_dicts'][0]  # list length batch
    hm = pred0['hm']              # (num_classes, H, W) or (B?,C,H,W) depending on impl

    print(hm.requires_grad)
    # make sure it's a scalar and has gradient
    return hm.float().mean()

In [44]:
import os
import gc
import pickle
import torch
import numpy as np
from torch.cuda.amp import autocast
from pcdet.models import load_data_to_gpu

i = 10
dataset = test_set
#model defined above

idx = i
sample = dataset[idx]

if 'gt_boxes' not in sample or sample['gt_boxes'].shape[0] == 0:
        raise RuntimeError("No GT boxes in this frame; pick another index.")
        
gt_boxes_np = sample['gt_boxes']
box_dim = gt_boxes_np.shape[1]
boxes_xyzlhw = gt_boxes_np[:, :7]
cls_idx_np = gt_boxes_np[:, -1].astype(np.int64)

classes = dataset.class_names
vehicle_class_name = 'Vehicle'
cls_vehicle = classes.index(vehicle_class_name) + 1 #OpenPCDet uses 1-based label in gt boxes
vehicle_mask = (cls_idx_np == cls_vehicle)
bten = torch.from_numpy(boxes_xyzlhw).float().to('cuda') # (M, 7)

#distance to origin
ranges = torch.linalg.norm(bten[:, :3], dim = 1)
# Among vehicles only, choose closest
vehicle_indices = torch.nonzero(torch.from_numpy(vehicle_mask), as_tuple=False).squeeze(1)
victim_rel_idx = torch.argmin(ranges[vehicle_indices]).item()
victim_idx = vehicle_indices[victim_rel_idx].item()

victim = bten[victim_idx].to('cuda')    # (7,)
xa, ya, za_center, la, wa, ha, yawa = victim.tolist()

z_top = za_center + ha / 2.0
za = z_top 

label_id = cls_vehicle - 1

n = 300
omega_min_offset = 3
alpha, omega, R = init_spoof_spherical(
    xa, ya, za,
    la,
    omega_min_offset,
    n=n,
    random_=True
)

# print("ID R:", id(R))
ref7 = victim.clone()  # (7,) victim GT in lidar coords

lr    = 0.05
steps = 200
R_min = 2.0
R_max = 80.0
eps_i = 0.1
eps_s = 0.1

device = next(model.parameters()).device
alpha = alpha.to(device)
omega = omega.to(device)
R     = R.to(device).requires_grad_(True)

print("R is leaf:", R.is_leaf, "requires_grad:", R.requires_grad, "grad_fn:", R.grad_fn)
R.retain_grad()
optim = torch.optim.Adam([R], lr=lr)
hidden_count     = 0
hidden_count_lb  = 1
best_score_ub    = 0.10
best_iou_ub      = 0.75
loss_list        = []
num_rel_list     = []
batch = dataset.collate_batch([sample])   # numpy arrays
load_data_to_gpu(batch)                  # in-place → torch.cuda tensors

optim.zero_grad(set_to_none=True)
spoof_xyz = spherical_to_xyz(alpha, omega, R,xa, ya, za,la, ha, wa, yawa)
spoof_i = make_intensities_elongation_like(spoof_xyz, value_i=0.7, value_e = 0.05)
b_idx = torch.full((spoof_i.shape[0], 1), 0).to(device)
spoof_xyz_i = torch.cat((b_idx, spoof_xyz, spoof_i), dim = 1)
points_real = batch['points']# (n, 1)
points_diff = torch.cat([points_real.detach(), spoof_xyz_i], dim=0)

spoof_xyz.retain_grad()
spoof_xyz_i.retain_grad()
points_diff.retain_grad()

with torch.enable_grad():
    model.eval()
    hm_raw, center_raw, dim_raw, rot_raw, vox_proxy, packed = forward_with_ste_centerpoint_raw(model, batch, points_diff)
    print("hm_raw:", hm_raw.requires_grad, hm_raw.grad_fn)
    print("packed:", packed.requires_grad, packed.grad_fn)
    print("vox_proxy:", vox_proxy.requires_grad, vox_proxy.grad_fn)
    loss = hm_raw.reshape(-1).mean()
    loss.backward()
    print("R.grad is None?", R.grad is None)
    print("R.grad mean abs:", None if R.grad is None else R.grad.abs().mean().item())
    print("spoof_xyz.grad None?", spoof_xyz.grad is None)
    print("spoof_xyz_i.grad None?", spoof_xyz_i.grad is None)
    print("points_diff.grad None?", points_diff.grad is None)



R is leaf: True requires_grad: True grad_fn: None
Spoof points voxelized: 298
Spoof points used in voxel_features: 298
keys after vfe: ['voxels', 'voxel_coords', 'voxel_num_points', 'voxel_features']
voxel_features: (True, <DivBackward0 object at 0x1554100e9610>, torch.Size([76621, 5]))
keys after backbone3d: ['batch_size', 'points', 'voxels', 'voxel_coords', 'voxel_num_points', 'voxel_features', 'encoded_spconv_tensor', 'encoded_spconv_tensor_stride', 'multi_scale_3d_features', 'multi_scale_3d_strides']
Grad inside Backbone3D (SparseTensor): True
vox_proxy requires_grad: True <AddBackward0 object at 0x1554100e9610>
packed requires_grad: True <ViewBackward0 object at 0x1554100e9610>
spatial_features_2d requires_grad: True <CatBackward0 object at 0x1554100e9610>
x requires_grad: True <ReluBackward0 object at 0x1554100e9610>
hm_raw: True <ConvolutionBackward0 object at 0x15552c438340>
packed: True <ViewBackward0 object at 0x15552c438340>
vox_proxy: True <AddBackward0 object at 0x15552c43

In [38]:
vox_proxy, _, _, _, packed = build_voxels_ste(model, points_diff)
packed.retain_grad()
points_diff.retain_grad()

# emulate MeanVFE explicitly
vf = vox_proxy.sum(dim=1)   # (V, C)
loss = vf.sum()
loss.backward()

print("packed.grad None?", packed.grad is None)
print("points_diff.grad None?", points_diff.grad is None)
print("R.grad None?", R.grad is None)

packed.grad None? False
points_diff.grad None? False
R.grad None? False


In [23]:
R.requires_grad

True

In [14]:
for t in range(1):
    optim.zero_grad(set_to_none=True)
    spoof_xyz = spherical_to_xyz(alpha, omega, R,xa, ya, za,la, ha, wa, yawa)
    
    spoof_i = make_intensities_elongation_like(spoof_xyz, value_i=0.7, value_e = 0.05)  # (n, 1)
    b_idx = torch.full((spoof_i.shape[0], 1), 0).to(device)
    spoof_xyz_i = torch.cat((b_idx, spoof_xyz, spoof_i), dim = 1)
    points_real = batch['points']
    
    points_diff = torch.cat([points_real.detach(), spoof_xyz_i], dim=0)
    # points_diff.requires_grad_(True)          # optional; leaf params are your R/alpha/etc
    # if you want to inspect spoof_points grad:
    spoof_xyz_i.retain_grad()
    R.retain_grad()
    
    print("grad enabled at call site:", torch.is_grad_enabled())
    with torch.enable_grad():
        # print("ID R:", id(R))
        # print("ID spoof tensor:", id(spoof_xyz_i))
        # batch_out, hm, vox_proxy, packed, vox_in = forward_with_ste_centerpoint(model, batch, points_diff)
        # loss = hm.float().mean()
        # print("loss.grad:", loss.grad)
        
        torch.set_grad_enabled(True)   # just to be safe

        model.eval()                   # eval is fine (BN/dropout behavior). Do NOT use inference_mode.

        hm_raw, center_raw, dim_raw, rot_raw, vox_proxy, packed = forward_with_ste_centerpoint_raw(model, batch, points_diff)

        print("hm_raw:", hm_raw.requires_grad, hm_raw.grad_fn)
        loss = hm_raw.float().mean()

        loss.backward()
        print("R.grad is None?", R.grad is None)
        print("R.grad mean abs:", None if R.grad is None else R.grad.abs().mean().item())
        g = torch.autograd.grad(loss, vox_proxy, retain_graph=True, allow_unused=True)[0]
        print("grad(loss, vox_proxy) is None?", g is None)
      

    # loss.backward()
    print("ID R:", id(R))
    print("ID spoof tensor:", id(spoof_xyz_i))
    print("R.grad:", R.grad)
   

ID R: 23450467139152
R is leaf: True requires_grad: True grad_fn: None
grad enabled at call site: True
hm_raw: True <ConvolutionBackward0 object at 0x1553f8b83d00>
R.grad is None? True
R.grad mean abs: None
grad(loss, vox_proxy) is None? True
ID R: 23450467139152
ID spoof tensor: 23450543053696
R.grad: None


In [15]:
#     pc_range = torch.tensor(cfg.DATA_CONFIG.POINT_CLOUD_RANGE,
#                         device=points.device)
#     voxel_size = torch.tensor(cfg.DATA_CONFIG.DATA_PROCESSOR[2].VOXEL_SIZE,
#                           device=points.device)
#     xyz = points[:, 1:4]
#     # voxel_xyz = torch.floor((xyz - pc_range[:3]) / voxel_size).long()
#     voxel_xyz_float = (xyz - pc_range[:3]) / voxel_size
#     voxel_xyz = voxel_xyz_float.detach().floor() + (voxel_xyz_float - voxel_xyz_float.detach())
#     voxel_xyz = voxel_xyz.long()
#     grid_size = ((pc_range[3:] - pc_range[:3]) / voxel_size).long()
#     grid_x, grid_y, grid_z = grid_size.tolist()
#     valid_mask = (
#         (voxel_xyz[:, 0] >= 0) &
#         (voxel_xyz[:, 1] >= 0) &
#         (voxel_xyz[:, 2] >= 0) &
#         (voxel_xyz[:, 0] < grid_x) &
#         (voxel_xyz[:, 1] < grid_y) &
#         (voxel_xyz[:, 2] < grid_z)
#     )

#     points = points[valid_mask]
#     voxel_xyz = voxel_xyz[valid_mask]
#     batch_idx = points[:, 0].long().unsqueeze(1)
#     voxel_coords = torch.cat(
#         [batch_idx,
#          voxel_xyz[:, 2:3],  # z
#          voxel_xyz[:, 1:2],  # y
#          voxel_xyz[:, 0:1]], # x
#         dim=1
#     )
#     unique_coords, inverse = torch.unique(
#         voxel_coords, dim=0, return_inverse=True
#     )
#     num_voxels = unique_coords.shape[0]
#     num_points = torch.zeros(num_voxels, device=points.device)
#     point_features = points[:, 1:] 
#     voxel_features = torch.zeros(
#         num_voxels, point_features.shape[1],
#         device=points.device
#     )
#     voxel_features.index_add_(0, inverse, point_features)

#     num_points.index_add_(0, inverse,
#                       torch.ones_like(inverse, dtype=torch.float))
#     voxel_features = voxel_features / num_points.clamp(min=1).unsqueeze(1)
#     batch_dict = {
#         'voxels': voxel_features.unsqueeze(1),  # (M, T=1, C)
#         'voxel_coords': unique_coords,
#         'voxel_num_points': num_points.long(),
#         'batch_size': 1,
#     }
#     batch_dict['voxel_coords'] = batch_dict['voxel_coords'].int().contiguous()
#     batch_dict['voxels'] = batch_dict['voxels'].float().contiguous()
#     batch_dict['voxel_num_points'] = batch_dict['voxel_num_points'].int().contiguous()
#     def forward_from_voxel_features(model, voxel_features, voxel_coords, batch_size=1):
#         bd = {
#             "voxel_features": voxel_features,          # (V, C)
#             "voxel_coords": voxel_coords.int(),        # (V, 4) [b,z,y,x]
#             "batch_size": batch_size,
#         }
#         bd = model.backbone_3d(bd)
#         bd = model.map_to_bev_module(bd)
#         bd = model.backbone_2d(bd)
#         bd = model.dense_head(bd)
#         return bd
#     voxel_coords, voxel_features = ste_voxelize_mean(points, voxel_size, pc_range)
#     bd = forward_from_voxel_features(model, voxel_features, voxel_coords, batch_size = 1)
#     p0 = model.dense_head.forward_ret_dict["pred_dicts"][0]
#     heatmap = p0.get("hm", None)  # inspect keys if different
    
#     loss = heatmap.sum()
#     loss.backward()
    
#     print("R.grad:", R.grad)
#     print("R.grad is None:", R.grad is None)
#     print("R.grad norm:", None if R.grad is None else R.grad.norm())

In [16]:
  batch_dict = model.vfe(batch_dict)
    batch_dict = model.backbone_3d(batch_dict)
    batch_dict = model.map_to_bev_module(batch_dict)
    batch_dict = model.backbone_2d(batch_dict)
    batch_dict = model.dense_head(batch_dict)
    
    frd = model.dense_head.forward_ret_dict
    print(frd.keys())
    print(type(frd.get("pred_dicts", None)))
    if "pred_dicts" in frd:
        p0 = frd["pred_dicts"][0] if isinstance(frd["pred_dicts"], list) else frd["pred_dicts"]
        print("pred_dicts[0] keys:", p0.keys())

IndentationError: unexpected indent (2213116239.py, line 2)

In [ ]:
    with autocast(dtype=torch.bfloat16):
        # boxes, scores, logits = forward_with_ste(model, batch_spoof, n)
        boxes, scores, logits, spoof_proxy, slot_to_point, p2v = forward_with_ste(model, batch, spoof_ids)
        ious = iou_bev(boxes, ref7)
        # detected, best_Score = object_still_detected
        best_iou = ious.max().item()
     
        # z = logits[:, label_id]
        z = logits
        loss, num_rel = hiding_loss(z, ious, eps_i=eps_i, eps_s=eps_s)
    print("--- gradient debug ---")
    print("torch.is_grad_enabled:", torch.is_grad_enabled())
    print("R.requires_grad:", R.requires_grad)
    print("spoof_xyz.requires_grad:", spoof_xyz.requires_grad)
    print("points.requires_grad:", batch_spoof['points'][0].requires_grad)
    print("logits.requires_grad:", logits.requires_grad)
    print("loss.requires_grad:", loss.requires_grad)
    print("loss.grad_fn:", loss.grad_fn)
    loss.backward()
    print("spoof_proxy.grad: ", spoof_proxy.grad)
    print("R.grad:", R.grad)
    optim.step()
    
    

In [ ]:
# SANITY TEST: STE voxelization ONLY

torch.set_grad_enabled(True)

# fake spoof parameter
R = torch.randn(1, device='cuda', requires_grad=True)

# fake spoof point depends on R
spoof_xyz = torch.stack([
    R + 1.0,
    R * 2.0,
    R * 3.0
], dim=1)                 # (1,3)

spoof_i = torch.ones((1,1), device='cuda')
spoof_point = torch.cat([torch.zeros((1,1), device='cuda'), spoof_xyz, spoof_i], dim=1)
spoof_point.retain_grad()

# fake real point (detached)
real_point = torch.tensor([[0, 1.0, 2.0, 3.0, 0.5]], device='cuda')

points = torch.cat([real_point, spoof_point], dim=0)

vox_proxy, _, _, _, packed = build_voxels_ste(model, points)

print("packed.requires_grad:", packed.requires_grad)
print("vox_proxy.requires_grad:", vox_proxy.requires_grad)

loss = vox_proxy.sum()
loss.backward()

print("R.grad:", R.grad)
print("spoof_point.grad:", spoof_point.grad)


In [ ]:
import plotly.io as pio
import numpy as np
import plotly.graph_objects as go
import pickle
import os

import plotly.express as px
import numpy as np
import torch
import pandas as pd
import plotly.io as pio

def plot_batch_spoof(batch_spoof, size=2, color_by='intensity', title="batch_spoof point cloud"):
    """
    Plot the point cloud inside batch_spoof['inputs']['points'][0].
    color_by: one of 'intensity' or 'z' or 'distance' or 'none'
    size: marker size
    """
    # Get points tensor (expected shape: (N,4) where cols are x,y,z,intensity)
    pts = batch_spoof['points']
    # If on GPU move to cpu
    if isinstance(pts, torch.Tensor):
        pts_np = pts.detach().cpu().numpy()
    else:
        pts_np = np.asarray(pts)

    if pts_np.ndim != 2 or pts_np.shape[1] < 3:
        raise ValueError("Expected point array shape (N,>=3). Found: %s" % (pts_np.shape,))

    x, y, z = pts_np[:, 1], pts_np[:, 2], pts_np[:, 3]
    # intensity column if available
    intensity = pts_np[:, 4] if pts_np.shape[1] > 3 else None
    distance = np.linalg.norm(pts_np[:, 1:4], axis=1)

    if color_by == 'intensity' and intensity is not None:
        color = intensity
        color_title = 'intensity'
    elif color_by == 'z':
        color = z
        color_title = 'z'
    elif color_by == 'distance':
        color = distance
        color_title = 'distance'
    else:
        color = None
        color_title = ''

    df = pd.DataFrame({'x': x, 'y': y, 'z': z})
    if color is not None:
        df[color_title] = color

    fig = px.scatter_3d(
        df,
        x='x', y='y', z='z',
        color=color_title if color is not None else None,
        title=f"{title} (color: {color_title})" if color_title else title,
        hover_data=[color_title] if color_title else None
    )

    # Set marker size
    fig.update_traces(marker=dict(size=size))

    # Compute real-world scale and set non-equal aspect ratio
    xrange = np.max(x) - np.min(x)
    yrange = np.max(y) - np.min(y)
    zrange = np.max(z) - np.min(z)

    # Avoid divisions by zero
    xrange = max(xrange, 1e-6)
    yrange = max(yrange, 1e-6)
    zrange = max(zrange, 1e-6)

    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X', range=[np.min(x), np.max(x)]),
            yaxis=dict(title='Y', range=[np.min(y), np.max(y)]),
            zaxis=dict(title='Z', range=[np.min(z), np.max(z)]),
            aspectmode='manual',                # key: don't make cube
            aspectratio=dict(
                x=xrange / max(zrange, yrange, xrange),
                y=yrange / max(zrange, yrange, xrange),
                z=zrange / max(zrange, yrange, xrange)
            )
        ),
        margin=dict(l=0, r=0, t=30, b=0)
    )

    fig.show()

In [ ]:
pio.renderers.default = "iframe_connected"   # Works in most remote Jupyter setups
# plot_batch_spoof(batch)

In [ ]:
plot_batch_spoof(batch_spoof)

In [ ]:
# def forward_with_ste_centerpoint(model, base_batch_dict, points_diff):
#     """
#     base_batch_dict: your original PCDet batch_dict (for metadata etc)
#     points_diff: (N, 1+C) with spoof rows differentiable
#     """
#     # 1) Build STE voxels
#     vox_proxy, voxel_coords, voxel_num_points, _, packed = build_voxels_ste(model, points_diff)

#     print("DEBUG packed.requires_grad:", packed.requires_grad, "packed.grad_fn:", packed.grad_fn)
#     print("DEBUG vox_proxy.requires_grad:", vox_proxy.requires_grad, "vox_proxy.grad_fn:", vox_proxy.grad_fn)


#     batch_dict = {}
#     batch_dict['batch_size'] = int(base_batch_dict.get('batch_size', 1))

#     batch_dict['points'] = points_diff# keep for debugging
#     vox_in = vox_proxy.contiguous().float()
#     batch_dict['voxels'] = vox_proxy.contiguous().float()
#     batch_dict['voxel_coords'] = voxel_coords.contiguous().int()
#     batch_dict['voxel_num_points'] = voxel_num_points.contiguous().int()
#     _rg(batch_dict['voxels'], "input voxels")
#     # 3) Normal CenterPoint modules (like you already do)
#     model.eval()
#     # for p in model.parameters():
#     #     p.requires_grad_(False)

#     batch_dict = model.vfe(batch_dict)
#     _rg(batch_dict.get('voxel_features', None), "after vfe voxel_features")
#     batch_dict = model.backbone_3d(batch_dict)
#     _rg(batch_dict.get('spatial_features', None), "after backbone_3d spatial_features")
#     batch_dict = model.map_to_bev_module(batch_dict)
#     _rg(batch_dict.get('spatial_features_2d', None), "after bev spatial_features_2d")
#     batch_dict = model.backbone_2d(batch_dict)
#     _rg(batch_dict.get('spatial_features_2d', None), "after backbone_2d spatial_features_2d")
# #     batch_dict = model.dense_head(batch_dict)
    
# #     hm = model.dense_head.forward_ret_dict['pred_dicts'][0]['hm']
# #     _rg(hm, "hm")
    
#     spatial_features_2d = batch_dict['spatial_features_2d']

#     # shared conv
#     x = model.dense_head.shared_conv(spatial_features_2d)

#     # run the first (and only) SeparateHead
#     head = model.dense_head.heads_list[0]
#     hm     = head.hm(x)
#     center = head.center(x)
#     dim    = head.dim(x)
#     rot    = head.rot(x)

#     return batch_dict, hm, vox_proxy, packed, vox_in